In [ ]:
TOLERANCE = 5
MIN_READS = 1   

def load_sj(filename):
    junctions = []
    with open(filename) as f:
        for line in f:
            parts = line.strip().split("\t")
            chr = parts[0]
            start = int(parts[1])
            end = int(parts[2])
            strand = parts[3]
            unique_reads = int(parts[6])
            junctions.append((chr, start, end, strand, unique_reads))
    return junctions
sj_wt = load_sj("WT_SJ.out.tab")
sj_ko = load_sj("KO_SJ.out.tab")
strand_map = {"+": "1", "-": "2"}
results = []
with open("optional_exons.bed") as f:
    for line in f:
        parts = line.strip().split("\t")
        chr = parts[0]
        start = int(parts[1])
        end = int(parts[2])
        gene = parts[3]
        strand = parts[5]
        star_strand = strand_map[strand]
        def compute_psi(sj_data):
            inclusion = 0
            skipping = 0
            for (j_chr, j_start, j_end, j_strand, reads) in sj_data:
                if j_chr != chr:
                    continue
                if j_strand != star_strand:
                    continue
                # Inclusion links (Exon1 -> Exon start)
                if abs(j_end - start) <= TOLERANCE:
                    inclusion += reads

                # Inclusion rechts (Exon end -> Exon3)
                if abs(j_start - end) <= TOLERANCE:
                    inclusion += reads

                # Skipping Junction (über Exon hinweg)
                if j_start < start and j_end > end:
                    skipping += reads

            total = inclusion + skipping

            if total < MIN_READS:
                return None   # nicht exprimiert

            return inclusion / total

        psi_wt = compute_psi(sj_wt)
        psi_ko = compute_psi(sj_ko)

        # Nur behalten wenn beide exprimiert
        if psi_wt is not None and psi_ko is not None:
            delta = psi_wt - psi_ko
            results.append((gene, start, end, psi_wt, psi_ko, delta))
with open("PSI_WT_vs_KO_filtered.csv", "w") as out:
    out.write("gene,exon_start,exon_end,PSI_WT,PSI_KO,delta_PSI\n")
    for r in results:
        out.write(f"{r[0]},{r[1]},{r[2]},{round(r[3],4)},{round(r[4],4)},{round(r[5],4)}\n")
print("Fertig")